# NPS Heatmap Analysis

This notebook is the **main navigation point** for building an NPS heatmap from many `.npz` survey files.

It includes:
- RFI masking and patching controls
- Interpolation strength control (`epsilon`)
- Frequency-band selection
- One-cell execution to produce and preview the map

`nps_heatmap.py` is kept as a reusable module, but the run workflow lives here.

## Config Cells


In [ ]:
from pathlib import Path
import importlib
import matplotlib.pyplot as plt
import nps_heatmap

# Ensure notebook always uses latest edits from nps_heatmap.py
importlib.reload(nps_heatmap)
run = nps_heatmap.run

# -------- USER CONFIG --------
NPZ_GLOB = "nps_data_4_29_final/*.npz"      # change to your survey location if needed
OUT_PNG = "nps_heatmap_rfi_clean.png"

# Galactic map bounds
LMIN, LMAX = 210.0, 380.0   # 380 means wrapped through l=20 deg
BMIN, BMAX = 0.0, 90.0
GRID_STEP_DEG = 1.0

# Interpolation strength (main smoothing knob)
EPSILON_DEG = .8

# Bandpower frequency window (set to None for full band)
FMIN_HZ = 1.4190e9   # ~-200 km/s from HI rest
FMAX_HZ = 1.4220e9   # +200 km/s from HI rest
MODE = "A+B"  # options: "A+B", "A", "B"

METRIC = "peak_excess"
BASELINE_WIDTH = 31

# RFI controls
RFI_SIGMA = 5.5
RFI_EDGE_TRIM = 12
RFI_PATCH = True
RFI_RANGES_HZ = [
    # (1.42032e9, 1.42037e9),
    # (1.42077e9, 1.42083e9),
]

print("Configuration loaded.")

In [ ]:
import numpy as np
import glob

C_KMS = 299792.458
F_HI  = 1420.40575177e6

# ── 1. Load and average the last 4 cal files for bandpass ────────────────────
CAL_GLOB  = 'nps_data_4_29_final/cal/*.npz'
cal_files = sorted(glob.glob(CAL_GLOB))[-4:]
print(f"Found {len(cal_files)} cal files:")
for f in cal_files:
    print(f"  {f.split('/')[-1]}")

if len(cal_files) == 0:
    raise FileNotFoundError(f"No cal files matched: {CAL_GLOB}")

bp_A_stack, bp_B_stack, fA_cal = [], [], None

for fn in cal_files:
    z  = np.load(fn, allow_pickle=True)
    fA = np.asarray(z['freq_hz_A']).ravel()
    if fA_cal is None:
        fA_cal = fA
    n  = fA.size

    def gspec(key):
        arr = np.asarray(z[key], dtype=float)
        if arr.ndim == 2:
            return arr[:, 0][:n]
        return arr[:n]

    sA = 0.5 * (gspec('spec_A_pol0') + gspec('spec_A_pol1'))
    sB = 0.5 * (gspec('spec_B_pol0') + gspec('spec_B_pol1'))
    m  = min(sA.size, sB.size, n)
    bp_A_stack.append(sA[:m])
    bp_B_stack.append(sB[:m])

# Pad to same length then average
max_len = max(a.size for a in bp_A_stack)
def pad(a): return np.pad(a, (0, max_len - a.size), constant_values=np.nan)

bp_A = np.nanmean(np.stack([pad(a) for a in bp_A_stack]), axis=0)
bp_B = np.nanmean(np.stack([pad(b) for b in bp_B_stack]), axis=0)

valid_A = bp_A[bp_A > 0]
valid_B = bp_B[bp_B > 0]
bp_A_norm = bp_A / np.nanmedian(valid_A)
bp_B_norm = bp_B / np.nanmedian(valid_B)

print(f"\nBandpass flatness after averaging {len(cal_files)} files:")
print(f"  A: std/mean = {valid_A.std()/valid_A.mean()*100:.2f}%")
print(f"  B: std/mean = {valid_B.std()/valid_B.mean()*100:.2f}%")

# ── 2. Measure fs_peak at highest-b survey pointings for LAB comparison ───────
print("\n=== Highest-b pointings for LAB calibration ===")
print("Visit: https://www.astro.uni-bonn.de/hisurvey/AllSky_profiles/")
print(f"{'l':>7}  {'b':>7}  {'fs_peak':>10}  T_sys = T_b_LAB / fs_peak")
print("-" * 55)

results = []
for fn in sorted(glob.glob(NPZ_GLOB)):
    try:
        z  = np.load(fn, allow_pickle=True)
        fA = np.asarray(z['freq_hz_A']).ravel()
        n  = fA.size

        def gspec(key):
            arr = np.asarray(z[key], dtype=float)
            if arr.ndim == 2:
                return arr[:, 0][:n] if arr.shape[1] > 1 else arr.ravel()[:n]
            return arr[:n]

        sA = 0.5 * (gspec('spec_A_pol0') + gspec('spec_A_pol1'))
        sB = 0.5 * (gspec('spec_B_pol0') + gspec('spec_B_pol1'))
        m  = min(fA.size, sA.size, sB.size)
        fs = (sA[:m] - sB[:m]) / sB[:m]
        v  = C_KMS * (F_HI - fA[:m]) / F_HI

        win = (np.abs(v) <= 80) & np.isfinite(fs)
        if not win.any():
            continue
        results.append({
            'l': float(z['l_deg']),
            'b': float(z['b_deg']),
            'fs_peak': float(np.nanmax(fs[win])),
        })
    except Exception:
        continue

results.sort(key=lambda r: -r['b'])
for r in results[:10]:
    print(f"  {r['l']:5.1f}    {r['b']:5.1f}    {r['fs_peak']:8.5f}"
          f"    = T_b_LAB / {r['fs_peak']:.5f}")

# ── 3. Enter LAB T_peak values after lookup, then rerun ──────────────────────
lab_measurements = [
    # (l_deg, b_deg, T_peak_K_from_LAB)
     (210.0, 88.0, 2.0),
     (324.6, 88.0, 1.7),
     (353.4, 86.0, 1.5),
     (210.0, 84.0, 2.3),
     (248.3, 84.0, 2.3)
]

if lab_measurements:
    tsys_vals = []
    for l_l, b_l, T_lab in lab_measurements:
        closest = min(results, key=lambda r: (r['l']-l_l)**2 + (r['b']-b_l)**2)
        ts = T_lab / closest['fs_peak']
        tsys_vals.append(ts)
        print(f"\n  LAB: l={l_l}, b={b_l}, T_b={T_lab} K → T_sys = {ts:.1f} K")
    T_SYS_K = float(np.median(tsys_vals))
    label    = f"T_b (K)   [T_sys = {T_SYS_K:.0f} K from LAB]"
    print(f"\n  → T_SYS_K = {T_SYS_K:.1f} K")
else:
    T_SYS_K = 1.0
    label    = "Peak brightness  (T_b / T_sys,  dimensionless)"
    print("\n  No LAB measurements — plotting in relative units.")
    print("  Fill in lab_measurements above and rerun to get Kelvin.")

print(f"\n  T_SYS_K = {T_SYS_K}")
print(f"  Label:   '{label}'")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

C_KMS = 299792.458
F_HI  = 1420.40575177e6

# Pick the pointing closest to your LAB measurement
TARGET_L, TARGET_B = 210.0, 88.0

# Find the matching file
import glob
best_file, best_dist = None, 999
for fn in sorted(glob.glob(NPZ_GLOB)):
    try:
        z = np.load(fn, allow_pickle=True)
        l = float(z['l_deg']); b = float(z['b_deg'])
        d = (l - TARGET_L)**2 + (b - TARGET_B)**2
        if d < best_dist:
            best_dist, best_file = d, fn
    except Exception:
        continue

print(f"Closest file: {best_file.split('/')[-1]}")
z  = np.load(best_file, allow_pickle=True)
fA = np.asarray(z['freq_hz_A']).ravel()
n  = fA.size

def gspec(key):
    arr = np.asarray(z[key], dtype=float)
    if arr.ndim == 2: return arr[:, 0][:n]
    return arr[:n]

sA = 0.5 * (gspec('spec_A_pol0') + gspec('spec_A_pol1'))
sB = 0.5 * (gspec('spec_B_pol0') + gspec('spec_B_pol1'))
m  = min(sA.size, sB.size, n)
fA, sA, sB = fA[:m], sA[:m], sB[:m]

# Raw fs spectrum
with np.errstate(invalid='ignore', divide='ignore'):
    fs_raw = (sA - sB) / sB
v = C_KMS * (F_HI - fA) / F_HI

# Baseline-subtracted fs (what nps_heatmap.py actually uses)
from scipy.ndimage import median_filter
win = (np.abs(v) <= 80) & np.isfinite(fs_raw)
fs_win = fs_raw[win]
v_win  = v[win]
baseline = median_filter(fs_win.astype(float), size=31, mode='nearest')
line = fs_win - baseline

fig, axes = plt.subplots(3, 1, figsize=(12, 10))

axes[0].plot(v, fs_raw, lw=0.8)
axes[0].axvline(0, color='k', lw=0.5, ls='--')
axes[0].set_xlim(-300, 300)
axes[0].set_title(f'Raw fs = (sA-sB)/sB  at l={TARGET_L}, b={TARGET_B}')
axes[0].set_ylabel('fs (raw)')
axes[0].set_xlabel('v_topo (km/s)')

axes[1].plot(v_win, fs_win, lw=0.8, label='fs in ±80 km/s window')
axes[1].plot(v_win, baseline, lw=1.5, color='r', label='running median baseline')
axes[1].legend()
axes[1].set_title('fs within ±80 km/s window and its baseline')
axes[1].set_ylabel('fs')
axes[1].set_xlabel('v_topo (km/s)')

axes[2].plot(v_win, line, lw=0.8)
axes[2].axhline(0, color='k', lw=0.5)
axes[2].set_title(f'Baseline-subtracted line  —  peak = {np.nanmax(line):.5f}')
axes[2].set_ylabel('fs - baseline')
axes[2].set_xlabel('v_topo (km/s)')

plt.tight_layout()
plt.savefig('fs_diagnostic.png', dpi=150)
plt.show()

print(f"\nfs_peak (raw nanmax, no baseline):       {float(np.nanmax(fs_raw[win])):.5f}")
print(f"fs_peak (after baseline subtraction):    {float(np.nanmax(line)):.5f}")
print(f"Ratio raw/corrected:                     {float(np.nanmax(fs_raw[win]))/float(np.nanmax(line)):.2f}x")
print(f"\nIf LAB T_peak = ??? K at this pointing:")
print(f"  T_sys (using raw fs_peak):             "
      f"{0:.1f} K  <- fill in")
print(f"  T_sys (using baseline-corrected peak): "
      f"{0:.1f} K  <- fill in")
print(f"\nEnter your LAB T_peak for l={TARGET_L}, b={TARGET_B} and rerun:")
T_LAB = 2.0   # <-- set this, e.g. T_LAB = 9.5
if T_LAB:
    print(f"  T_sys (raw):       {T_LAB / np.nanmax(fs_raw[win]):.1f} K")
    print(f"  T_sys (corrected): {T_LAB / np.nanmax(line):.1f} K")

In [ ]:
import numpy as np
import glob
import importlib
import nps_heatmap
importlib.reload(nps_heatmap)

C_KMS = 299792.458
F_HI  = 1420.40575177e6
files = sorted(glob.glob(NPZ_GLOB))

def get_spec(z, key, n_freq):
    arr = np.asarray(z[key], dtype=float)
    return nps_heatmap._spectrum_to_1d(arr, n_freq=n_freq)

# ── Step 1: measure fs noise on off-line channels ─────────────────────────────
print("=== Per-pointing fs noise ===")
sigma_list, fs_peak_list, l_list, b_list = [], [], [], []

for fn in files:
    try:
        z = np.load(fn, allow_pickle=True)
        fA = np.asarray(z['freq_hz_A']).ravel()
        n  = fA.size
        sA = 0.5 * (get_spec(z, 'spec_A_pol0', n) + get_spec(z, 'spec_A_pol1', n))
        sB = 0.5 * (get_spec(z, 'spec_B_pol0', n) + get_spec(z, 'spec_B_pol1', n))
        m  = min(fA.size, sA.size, sB.size)
        fA, sA, sB = fA[:m], sA[:m], sB[:m]

        fs = (sA - sB) / sB
        v  = C_KMS * (F_HI - fA) / F_HI

        # Off-line channels only (exclude ±30 km/s around HI rest)
        off = (np.abs(v) > 30.0) & (np.abs(v) < 200.0) & np.isfinite(fs)
        if off.sum() < 10:
            continue

        # On-line peak (within ±80 km/s)
        on  = (np.abs(v) <= 80.0) & np.isfinite(fs)
        if not on.any():
            continue

        sigma_list.append(float(np.nanstd(fs[off])))
        fs_peak_list.append(float(np.nanmax(fs[on])))
        l_list.append(float(np.asarray(z['l_deg']).ravel()[0]))
        b_list.append(float(np.asarray(z['b_deg']).ravel()[0]))
    except Exception as e:
        continue

sigma_arr   = np.array(sigma_list)
fs_peak_arr = np.array(fs_peak_list)
l_arr       = np.array(l_list)
b_arr       = np.array(b_list)

# ── Step 2: radiometer prediction (for comparison only — does NOT give T_sys) ─
z0        = np.load(files[0], allow_pickle=True)
delta_nu  = float(z0['sample_rate_hz']) / float(z0['n_fft'])
tau       = float(z0['nblocks_acc'])    * float(z0['sub_window_sec'])
sigma_rad = np.sqrt(2) / np.sqrt(delta_nu * tau)   # expected for (sA-sB)/sB

print(f"Radiometer-predicted sigma_fs : {sigma_rad:.5f}")
print(f"Measured median sigma_fs      : {np.median(sigma_arr):.5f}")
print(f"Excess noise factor           : {np.median(sigma_arr)/sigma_rad:.2f}x")

# ── Step 3: LAB survey comparison — your 8 highest-b pointings ───────────────
print("\n=== LAB calibration pointings ===")
print("Look up each (l, b) at https://www.astro.uni-bonn.de/hisurvey/AllSky_profiles/")
print("Get T_peak [K] from the profile, then T_sys = T_peak_lab / fs_peak_measured\n")
print(f"{'l_deg':>8}  {'b_deg':>8}  {'fs_peak':>10}  T_sys = T_b_lab / fs_peak")
print("-" * 55)

order = np.argsort(b_arr)[::-1]
for i in order[:8]:
    print(f"  {l_arr[i]:6.1f}    {b_arr[i]:6.1f}    {fs_peak_arr[i]:8.4f}    = T_b_lab / {fs_peak_arr[i]:.4f}")

# ── Step 4: enter LAB values here once looked up ─────────────────────────────
print("\n=== Enter your LAB T_peak values below ===")
# Fill this in after visiting the LAB survey for each pointing above.
# Format: (l_deg, b_deg, T_b_lab_K)
lab_measurements = [
    # (330.0, 80.0, 6.5),   # example — replace with real values
    # (270.0, 75.0, 8.2),
]

if lab_measurements:
    tsys_from_lab = []
    for l_lab, b_lab, T_lab in lab_measurements:
        # Find closest pointing in your data
        dist = np.sqrt((l_arr - l_lab)**2 + (b_arr - b_lab)**2)
        idx  = np.argmin(dist)
        ts   = T_lab / fs_peak_arr[idx]
        tsys_from_lab.append(ts)
        print(f"  l={l_lab}, b={b_lab}: T_lab={T_lab:.1f} K, "
              f"fs_peak={fs_peak_arr[idx]:.4f}, T_sys={ts:.1f} K")

    T_SYS_K = float(np.median(tsys_from_lab))
    print(f"\nMedian T_sys from LAB: {T_SYS_K:.1f} K")
    print(f"Update your config:  T_SYS_K = {T_SYS_K:.1f}")
else:
    print("  (fill in lab_measurements list above, then rerun this cell)")
    T_SYS_K = None

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.ndimage import median_filter
import glob

C_KMS = 299792.458
F_HI  = 1420.40575177e6
T_SYS_K = 47.9  # confirmed from calibration

# ── LAB calibration measurements ─────────────────────────────────────────────
lab_measurements = [
    (210.0, 88.0, 2.0),
     (324.6, 88.0, 1.7),
     (353.4, 86.0, 1.5),
     (210.0, 84.0, 2.3),
     (248.3, 84.0, 2.3)
]

# ── Load the primary calibration pointing (l=210, b=88) ──────────────────────
TARGET_L, TARGET_B = 210.0, 88.0
best_file, best_dist = None, 999
for fn in sorted(glob.glob(NPZ_GLOB)):
    try:
        z = np.load(fn, allow_pickle=True)
        d = (float(z['l_deg']) - TARGET_L)**2 + (float(z['b_deg']) - TARGET_B)**2
        if d < best_dist:
            best_dist, best_file = d, fn
    except Exception:
        continue

z  = np.load(best_file, allow_pickle=True)
fA = np.asarray(z['freq_hz_A']).ravel()
n  = fA.size

def gspec(key):
    arr = np.asarray(z[key], dtype=float)
    if arr.ndim == 2: return arr[:, 0][:n]
    return arr[:n]

sA = 0.5 * (gspec('spec_A_pol0') + gspec('spec_A_pol1'))
sB = 0.5 * (gspec('spec_B_pol0') + gspec('spec_B_pol1'))
m  = min(sA.size, sB.size, n)
fA, sA, sB = fA[:m], sA[:m], sB[:m]

with np.errstate(invalid='ignore', divide='ignore'):
    fs_raw = (sA - sB) / sB
v = C_KMS * (F_HI - fA) / F_HI

win      = (np.abs(v) <= 80) & np.isfinite(fs_raw)
fs_win   = fs_raw[win]
v_win    = v[win]
baseline = median_filter(fs_win.astype(float), size=31, mode='nearest')
line     = fs_win - baseline
fs_peak  = float(np.nanmax(line))

# ── Compute T_sys per pointing ────────────────────────────────────────────────
tsys_per_pointing = []
for fn in sorted(glob.glob(NPZ_GLOB)):
    try:
        z2 = np.load(fn, allow_pickle=True)
        fA2 = np.asarray(z2['freq_hz_A']).ravel(); n2 = fA2.size
        def gs2(k):
            a = np.asarray(z2[k], dtype=float)
            return (a[:,0][:n2] if a.ndim==2 else a[:n2])
        sA2 = 0.5*(gs2('spec_A_pol0')+gs2('spec_A_pol1'))
        sB2 = 0.5*(gs2('spec_B_pol0')+gs2('spec_B_pol1'))
        m2  = min(fA2.size, sA2.size, sB2.size)
        with np.errstate(invalid='ignore', divide='ignore'):
            fs2 = (sA2[:m2]-sB2[:m2])/sB2[:m2]
        v2 = C_KMS*(F_HI-fA2[:m2])/F_HI
        w2 = (np.abs(v2)<=80) & np.isfinite(fs2)
        if not w2.any(): continue
        bl2 = median_filter(fs2[w2].astype(float), size=31, mode='nearest')
        pk2 = float(np.nanmax(fs2[w2]-bl2))
        l2, b2 = float(z2['l_deg']), float(z2['b_deg'])
        for l_lab, b_lab, T_lab in lab_measurements:
            if (l2-l_lab)**2 + (b2-b_lab)**2 < 4:
                tsys_per_pointing.append((l2, b2, T_lab, pk2, T_lab/pk2))
    except Exception:
        continue

# ── Figure ────────────────────────────────────────────────────────────────────
plt.rcParams.update({
    'font.family':  'serif',
    'font.size':    11,
    'axes.labelsize': 12,
    'axes.titlesize': 12,
    'legend.fontsize': 10,
    'figure.dpi':   150,
})

fig = plt.figure(figsize=(12, 10))
gs  = gridspec.GridSpec(3, 2, figure=fig,
                        left=0.08, right=0.97,
                        top=0.93,  bottom=0.07,
                        hspace=0.45, wspace=0.35)

ax1 = fig.add_subplot(gs[0, :])   # full-width: raw fs
ax2 = fig.add_subplot(gs[1, 0])   # baseline panel
ax3 = fig.add_subplot(gs[1, 1])   # baseline-subtracted line
ax4 = fig.add_subplot(gs[2, :])   # T_sys calibration scatter

# Panel 1 — Raw fs spectrum
ax1.plot(v, fs_raw, lw=0.7, color='steelblue', alpha=0.85)
ax1.axvline(0, color='k', lw=0.8, ls='--', alpha=0.5, label='HI rest frame')
ax1.axvspan(-80, 80, alpha=0.08, color='orange', label='±80 km/s search window')
ax1.set_xlim(-300, 300)
ax1.set_ylim(-0.08, 0.10)
ax1.set_xlabel(r'$v_\mathrm{topo}$ (km s$^{-1}$)')
ax1.set_ylabel(r'$f_s = (s_A - s_B)\,/\,s_B$')
ax1.set_title(rf'Frequency-switched HI spectrum at $\ell={TARGET_L:.0f}°$, $b={TARGET_B:.0f}°$')
ax1.legend(loc='upper right', framealpha=0.9)
ax1.grid(True, alpha=0.3, lw=0.5)

# Panel 2 — ±80 km/s window with baseline
ax2.plot(v_win, fs_win, lw=0.8, color='steelblue', alpha=0.85, label=r'$f_s$')
ax2.plot(v_win, baseline, lw=2.0, color='firebrick', label='Running median baseline')
ax2.set_xlabel(r'$v_\mathrm{topo}$ (km s$^{-1}$)')
ax2.set_ylabel(r'$f_s$')
ax2.set_title(r'$f_s$ within ±80 km s$^{-1}$ window')
ax2.legend(framealpha=0.9)
ax2.grid(True, alpha=0.3, lw=0.5)

# Panel 3 — Baseline-subtracted line
ax3.plot(v_win, line, lw=0.8, color='steelblue', alpha=0.85)
ax3.axhline(0, color='k', lw=0.8)
ax3.axhline(fs_peak, color='firebrick', lw=1.2, ls='--',
            label=rf'Peak $f_s$ = {fs_peak:.4f}')
ax3.fill_between(v_win, 0, line, where=(line > 0),
                 alpha=0.25, color='steelblue')
ax3.set_xlabel(r'$v_\mathrm{topo}$ (km s$^{-1}$)')
ax3.set_ylabel(r'$f_s - f_{s,\mathrm{base}}$')
ax3.set_title('Baseline-subtracted HI line profile')
ax3.legend(framealpha=0.9)
ax3.grid(True, alpha=0.3, lw=0.5)

# Panel 4 — T_sys per LAB pointing
if tsys_per_pointing:
    ls_p   = [r[0] for r in tsys_per_pointing]
    bs_p   = [r[1] for r in tsys_per_pointing]
    tsys_p = [r[4] for r in tsys_per_pointing]
    tlab_p = [r[2] for r in tsys_per_pointing]
    fspk_p = [r[3] for r in tsys_per_pointing]

    sc = ax4.scatter(bs_p, tsys_p, c=ls_p, cmap='plasma',
                     s=120, zorder=5, edgecolors='k', lw=0.5)
    cb = fig.colorbar(sc, ax=ax4, pad=0.01)
    cb.set_label(r'Galactic longitude $\ell$ (deg)')

    T_median = float(np.median(tsys_p))
    ax4.axhline(T_median, color='firebrick', lw=1.5, ls='--',
                label=rf'Median $T_\mathrm{{sys}}$ = {T_median:.0f} K')
    ax4.axhspan(T_median-10, T_median+10, alpha=0.1, color='firebrick')

    # Annotate each point
    for l_p, b_p, tl, fp, ts in tsys_per_pointing:
        ax4.annotate(
            rf'$\ell$={l_p:.0f}°, $b$={b_p:.0f}°'
            rf'\n$T_{{b,\mathrm{{LAB}}}}$={tl:.2f} K',
            xy=(b_p, ts), xytext=(6, 4), textcoords='offset points',
            fontsize=8, color='0.3'
        )
else:
    # Fallback: just show the single-pointing result
    ax4.scatter([TARGET_B], [T_SYS_K], s=150, color='steelblue',
                edgecolors='k', lw=0.8, zorder=5)
    T_median = T_SYS_K
    ax4.axhline(T_median, color='firebrick', lw=1.5, ls='--',
                label=rf'$T_\mathrm{{sys}}$ = {T_median:.0f} K')
    ax4.annotate(
        rf'$\ell$={TARGET_L:.0f}°, $b$={TARGET_B:.0f}°'
        rf'\n$T_{{b,\mathrm{{LAB}}}}$ = 1.25 K'
        rf'\n$f_s$ peak = {fs_peak:.4f}',
        xy=(TARGET_B, T_SYS_K), xytext=(8, -20),
        textcoords='offset points', fontsize=9
    )

ax4.set_xlabel(r'Galactic latitude $b$ (deg)')
ax4.set_ylabel(r'$T_\mathrm{sys}$ (K)')
ax4.set_title(
    r'System temperature calibration via LAB survey '
    r'($T_\mathrm{sys} = T_{b,\mathrm{LAB}}\,/\,f_{s,\mathrm{peak}}$)'
)
ax4.legend(framealpha=0.9)
ax4.grid(True, alpha=0.3, lw=0.5)
ax4.set_ylim(0, max(tsys_p if tsys_per_pointing else [T_SYS_K]) * 1.4)

fig.suptitle(
    rf'Leuschner HI Receiver Calibration  —  '
    rf'$T_\mathrm{{sys}}$ = {T_median:.0f} K  '
    rf'(LAB survey cross-calibration)',
    fontsize=13, fontweight='bold'
)

plt.savefig('tsys_calibration_report.png', dpi=200, bbox_inches='tight')
plt.show()
print(f"Saved: tsys_calibration_report.png")
print(f"Final T_sys = {T_median:.1f} K")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.ndimage import median_filter
import glob

C_KMS = 299792.458
F_HI  = 1420.40575177e6

# ── LAB measurements: (l, b, T_peak_LAB_K, fs_peak_measured) ─────────────────
# Fill in your actual LAB T_peak values and measured fs_peaks from earlier output
lab_data = [
    (210.0, 88.0, 1.25, 0.04178),
    (324.6, 88.0, 2.10, 0.05120),  # replace with real LAB values
    (353.4, 86.0, 1.80, 0.03775),
    (210.0, 84.0, 1.95, 0.04573),
    (248.3, 84.0, 2.30, 0.05132),
]

ls_p    = np.array([r[0] for r in lab_data])
bs_p    = np.array([r[1] for r in lab_data])
tlab_p  = np.array([r[2] for r in lab_data])
fspk_p  = np.array([r[3] for r in lab_data])
tsys_p  = tlab_p / fspk_p
T_median = float(np.median(tsys_p))
T_std    = float(np.std(tsys_p))

# ── Single-column figure size (3.5" wide for 2-column paper) ─────────────────
plt.rcParams.update({
    'font.family':    'serif',
    'font.size':      8,
    'axes.labelsize': 9,
    'axes.titlesize': 9,
    'legend.fontsize': 7.5,
    'xtick.labelsize': 7.5,
    'ytick.labelsize': 7.5,
    'figure.dpi':     200,
})

import numpy as np
import glob
from scipy.ndimage import median_filter

C_KMS = 299792.458
F_HI  = 1420.40575177e6

def fs_peak_corrected(target_l, target_b, npz_glob='nps_data_4_29_final/*.npz'):
    best_file, best_dist = None, 1e9
    for fn in sorted(glob.glob(npz_glob)):
        z = np.load(fn, allow_pickle=True)
        try:
            l = float(z['l_deg'])
            b = float(z['b_deg'])
        except (TypeError, ValueError):
            continue   # skip cal files with None l/b
        d = (l - target_l)**2 + (b - target_b)**2
        if d < best_dist:
            best_dist, best_file = d, fn

    if best_file is None:
        raise RuntimeError(f"No valid pointings found in {npz_glob}")

    z  = np.load(best_file, allow_pickle=True)
    fA = np.asarray(z['freq_hz_A']).ravel(); n = fA.size
    def gs(k):
        a = np.asarray(z[k], dtype=float)
        return (a[:,0][:n] if a.ndim==2 else a[:n])
    sA = 0.5*(gs('spec_A_pol0')+gs('spec_A_pol1'))
    sB = 0.5*(gs('spec_B_pol0')+gs('spec_B_pol1'))
    m  = min(fA.size, sA.size, sB.size)
    with np.errstate(invalid='ignore', divide='ignore'):
        fs = (sA[:m]-sB[:m])/sB[:m]
    v  = C_KMS*(F_HI-fA[:m])/F_HI
    win = (np.abs(v) <= 80) & np.isfinite(fs)
    fs_w = fs[win]
    bl   = median_filter(fs_w.astype(float), size=31, mode='nearest')
    return float(np.nanmax(fs_w - bl))

# Recompute fs_peak (baseline-corrected) for each LAB pointing
lab_pointings = [
    (210.0, 88.0, 1.25),
    (324.6, 88.0, 2.10),
    (353.4, 86.0, 1.80),
    (210.0, 84.0, 1.95),
    (248.3, 84.0, 2.30),
]

print("Recomputing baseline-subtracted fs_peak for each LAB pointing:")
lab_data = []
for l, b, T_lab in lab_pointings:
    fs_pk = fs_peak_corrected(l, b)
    lab_data.append((l, b, T_lab, fs_pk))
    print(f"  l={l:6.1f}  b={b:5.1f}  T_lab={T_lab:.2f}  "
          f"fs_peak={fs_pk:.5f}  → T_sys = {T_lab/fs_pk:.1f} K")

fig, axes = plt.subplots(1, 2, figsize=(7.0, 3.0),
                         gridspec_kw={'wspace': 0.38})

# ── Left panel: T_sys per pointing scatter ────────────────────────────────────
ax = axes[0]
sc = ax.scatter(bs_p, tsys_p,
                c=ls_p, cmap='plasma',
                s=60, zorder=5,
                edgecolors='k', lw=0.5)

cb = fig.colorbar(sc, ax=ax, pad=0.02, aspect=20)
cb.set_label(r'$\ell$ (deg)', fontsize=8)
cb.ax.tick_params(labelsize=7)

ax.axhline(T_median, color='firebrick', lw=1.2, ls='--',
           label=rf'Median $T_\mathrm{{sys}}$ = {T_median:.0f} K')
ax.axhspan(T_median - T_std, T_median + T_std,
           alpha=0.12, color='firebrick', label=rf'$\pm1\sigma$ = {T_std:.0f} K')

ax.set_xlabel(r'Galactic latitude $b$ (deg)')
ax.set_ylabel(r'$T_\mathrm{sys}$ (K)')
ax.set_title(r'$T_\mathrm{sys}$ per calibration pointing')
ax.legend(loc='upper left', framealpha=0.9, fontsize=7)
ax.grid(True, alpha=0.3, lw=0.4)
ax.set_ylim(0, max(tsys_p) * 1.45)

# ── Right panel: LAB T_b vs measured fs_peak (calibration curve) ─────────────
ax2 = axes[1]
ax2.scatter(fspk_p, tlab_p,
            c=bs_p, cmap='viridis',
            s=60, zorder=5,
            edgecolors='k', lw=0.5,
            label='Calibration pointings')

# Best-fit line through origin: T_b = T_sys * fs_peak
fs_fit = np.linspace(0, max(fspk_p) * 1.2, 100)
ax2.plot(fs_fit, T_median * fs_fit,
         color='firebrick', lw=1.2, ls='--',
         label=rf'$T_b = T_\mathrm{{sys}} \cdot f_s$, $T_\mathrm{{sys}}$={T_median:.0f} K')

cb2 = fig.colorbar(
    plt.cm.ScalarMappable(
        cmap='viridis',
        norm=plt.Normalize(bs_p.min(), bs_p.max())
    ),
    ax=ax2, pad=0.02, aspect=20
)
cb2.set_label(r'$b$ (deg)', fontsize=8)
cb2.ax.tick_params(labelsize=7)

ax2.set_xlabel(r'Measured $f_{s,\mathrm{peak}}$')
ax2.set_ylabel(r'LAB $T_{b,\mathrm{peak}}$ (K)')
ax2.set_title(r'LAB cross-calibration curve')
ax2.legend(framealpha=0.9, fontsize=7)
ax2.grid(True, alpha=0.3, lw=0.4)
ax2.set_xlim(0, max(fspk_p) * 1.25)
ax2.set_ylim(0, max(tlab_p) * 1.35)

# ── Summary text box ──────────────────────────────────────────────────────────
summary = (
    rf'$N_\mathrm{{pointings}}$ = {len(lab_data)}'
    '\n'
    rf'$T_\mathrm{{sys}}$ = {T_median:.0f} $\pm$ {T_std:.0f} K'
    '\n'
    rf'$b$ range: {bs_p.min():.0f}°–{bs_p.max():.0f}°'
)
axes[1].text(0.97, 0.05, summary,
             transform=axes[1].transAxes,
             fontsize=7, va='bottom', ha='right',
             bbox=dict(boxstyle='round,pad=0.4',
                       facecolor='white', edgecolor='0.7', alpha=0.9))

fig.suptitle(
    r'System Temperature Calibration — LAB Survey Cross-Comparison',
    fontsize=9, fontweight='bold', y=1.01
)

plt.savefig('tsys_calibration_report.png',
            dpi=300, bbox_inches='tight')
plt.show()
print(f"Saved: tsys_calibration_report.png")
print(f"T_sys = {T_median:.1f} ± {T_std:.1f} K  ({len(lab_data)} pointings)")

In [ ]:
# Optional: sweep interpolation strengths for visual comparison
METRIC         = "peak_excess"
BASELINE_WIDTH = 31
EPSILON_DEG    = 0.2        # start here, adjust after sweep
FMIN_HZ        = 1.4190e9
FMAX_HZ        = 1.4220e9
MODE           = "A+B"

EPSILON_LIST = [0.1, 0.2, 0.3, 0.5]

for eps in EPSILON_LIST:
    out = f"nps_heatmap_eps_{eps:.2f}.png"
    run(
        npz_glob=NPZ_GLOB,
        output_png=out,
        fmin_hz=FMIN_HZ,
        fmax_hz=FMAX_HZ,
        spectrum_mode=MODE,
        epsilon_deg=eps,
        grid_step_deg=GRID_STEP_DEG,
        lmin=LMIN,
        lmax=LMAX,
        bmin=BMIN,
        bmax=BMAX,
        rfi_sigma_clip=RFI_SIGMA,
        rfi_edge_trim=RFI_EDGE_TRIM,
        rfi_patch=RFI_PATCH,
        rfi_ranges_hz=RFI_RANGES_HZ,
        metric=METRIC,
        baseline_width=BASELINE_WIDTH,
    )
    print(f"Saved: {out}")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

fig, axes = plt.subplots(2, 2, figsize=(18, 10))

for ax, eps in zip(axes.flatten(), EPSILON_LIST):
    img = mpimg.imread(f"nps_heatmap_eps_{eps:.2f}.png")
    ax.imshow(img)
    ax.axis("off")
    ax.set_title(f"epsilon = {eps}", fontsize=13)

fig.suptitle(f"Epsilon Sweep — metric={METRIC}", fontsize=15)
plt.tight_layout()
plt.show()

In [ ]:
from nps_heatmap import load_survey_points, interpolate_map, RfiConfig
import numpy as np
import matplotlib.pyplot as plt

rfi_cfg = RfiConfig(sigma_clip=RFI_SIGMA, edge_trim=RFI_EDGE_TRIM, patch_masked=RFI_PATCH)

points = load_survey_points(
    npz_glob=NPZ_GLOB,
    fmin_hz=FMIN_HZ,
    fmax_hz=FMAX_HZ,
    spectrum_mode=MODE,
    to_db=False,
    rfi_config=rfi_cfg,
    metric=METRIC,
    baseline_width=BASELINE_WIDTH,
)

ll, bb, zz = interpolate_map(
    points,
    lmin=LMIN, lmax=LMAX,
    bmin=BMIN, bmax=BMAX,
    grid_step_deg=GRID_STEP_DEG,
    epsilon_deg=EPSILON_DEG,
)

# Convert to radians for Mollweide — it expects longitude in [-pi, pi]
ll_rad = np.deg2rad(ll - 180.0)   # shift so galactic center is in the middle
bb_rad = np.deg2rad(bb)

vmin = float(np.nanpercentile(zz, 2))
vmax = float(np.nanpercentile(zz, 98))

fig = plt.figure(figsize=(14, 7))
ax = fig.add_subplot(111, projection="mollweide")
im = ax.pcolormesh(ll_rad, bb_rad, zz, shading="auto", cmap="magma", vmin=vmin, vmax=vmax)
fig.colorbar(im, ax=ax, orientation="horizontal", pad=0.05, label=f"{METRIC} (arb. units)")
ax.set_title(f"NPS Supershell — Mollweide projection (metric={METRIC}, ε={EPSILON_DEG})", pad=15)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("nps_mollweide.png", dpi=180)
plt.show()

## Heatmap With T_sys Calibration

In [ ]:
import importlib
import nps_heatmap
importlib.reload(nps_heatmap)

from nps_heatmap import load_survey_points, interpolate_map, plot_mollweide_nps_map, RfiConfig
import numpy as np

# ── 1. Set your T_sys from the LAB comparison (replace placeholder) ───────────
T_SYS_K = 150.0   # replace with your LAB-derived value

# ── 2. Load points using your existing config variables ───────────────────────
rfi_cfg = RfiConfig(
    sigma_clip=RFI_SIGMA,
    edge_trim=RFI_EDGE_TRIM,
    patch_masked=RFI_PATCH,
    manual_ranges_hz=tuple(RFI_RANGES_HZ),
)

points = load_survey_points(
    npz_glob=NPZ_GLOB,
    fmin_hz=FMIN_HZ,
    fmax_hz=FMAX_HZ,
    spectrum_mode=MODE,
    to_db=False,
    rfi_config=rfi_cfg,
    metric=METRIC,
    baseline_width=BASELINE_WIDTH,
)

# ── 3. Interpolate ────────────────────────────────────────────────────────────
ll, bb, zz_raw = interpolate_map(
    points,
    lmin=LMIN,
    lmax=LMAX,
    bmin=BMIN,
    bmax=BMAX,
    grid_step_deg=GRID_STEP_DEG,
    epsilon_deg=EPSILON_DEG,
)

# ── 4. Apply footprint mask (removes the unobserved gap) ──────────────────────
from nps_heatmap import _apply_footprint_mask
keep = _apply_footprint_mask(ll, bb, points, radius_deg=6.0)
zz_raw = np.where(keep, zz_raw, np.nan)

# ── 5. Convert to Kelvin ──────────────────────────────────────────────────────
zz_K = zz_raw * T_SYS_K
zz_K = np.where(zz_K < 0, np.nan, zz_K)   # drop any negative residuals

finite = zz_K[np.isfinite(zz_K)]
vmax_K = float(np.nanpercentile(finite, 98)) if finite.size else T_SYS_K

# ── 6. Plot Mollweide ─────────────────────────────────────────────────────────
fig_m, ax_m = plot_mollweide_nps_map(
    ll, bb, zz_K, points,
    title=f"NPS Supershell — Peak HI Brightness (ε={EPSILON_DEG})",
    cmap="magma",
    vmin=0.0,
    vmax=vmax_K,
    center_l_deg=300.0,
    colorbar_label=f"T_b (K)   [T_sys ≈ {T_SYS_K:.0f} K]",
)
fig_m.savefig("nps_brightness_kelvin_mollweide.png", dpi=180, bbox_inches="tight")
print(f"T_b range: {np.nanmin(zz_K):.1f} – {np.nanmax(zz_K):.1f} K")
print(f"Saved nps_brightness_kelvin_mollweide.png")

## Spur-Focused Run

If you did not see the earlier cell, use the code cell directly below this heading.

Tip: in Jupyter, use **Edit -> Find and Replace** and search for `nps_heatmap_spur_tuned`.

In [ ]:
from nps_heatmap import run
# Spur-focused run (duplicate, placed at notebook end for visibility)
METRIC = "peak_excess"
BASELINE_WIDTH = 151

out_path = run(
    npz_glob=NPZ_GLOB,
    output_png="nps_heatmap_spur_tuned.png",
    fmin_hz=FMIN_HZ,
    fmax_hz=FMAX_HZ,
    spectrum_mode=MODE,
    epsilon_deg=0.30,
    grid_step_deg=1.0,
    lmin=LMIN,
    lmax=LMAX,
    bmin=BMIN,
    bmax=BMAX,
    rfi_sigma_clip=RFI_SIGMA,
    rfi_edge_trim=RFI_EDGE_TRIM,
    rfi_patch=RFI_PATCH,
    rfi_ranges_hz=RFI_RANGES_HZ,
    metric=METRIC,
    baseline_width=BASELINE_WIDTH,
    interpolation_method="linear",
    smooth_sigma_deg=1.8,
    footprint_radius_deg=6.5,
)

print(f"Saved Cartesian: {Path(out_path).resolve()}")
print(f"Saved Mollweide: {(Path(out_path).with_name(Path(out_path).stem + '_mollweide' + Path(out_path).suffix)).resolve()}")

In [ ]:
# Doppler velocity map (LSR-corrected)
from nps_heatmap import run_velocity_map

vel_out = run_velocity_map(
    npz_glob=NPZ_GLOB,
    output_png="nps_velocity_lsr.png",
    fmin_hz=FMIN_HZ,
    fmax_hz=FMAX_HZ,
    spectrum_mode=MODE,
    grid_step_deg=1.0,
    lmin=LMIN,
    lmax=LMAX,
    bmin=BMIN,
    bmax=BMAX,
    rfi_sigma_clip=RFI_SIGMA,
    rfi_edge_trim=RFI_EDGE_TRIM,
    rfi_patch=RFI_PATCH,
    rfi_ranges_hz=RFI_RANGES_HZ,
    baseline_width=151,
    rest_freq_hz=1420.40575177e6,
    apply_lsr=True,
    # Leuschner defaults below (edit if needed)
    obs_lat_deg=37.9183,
    obs_lon_deg=-122.1537,
    obs_alt_m=304.0,
    interpolation_method="linear",
    epsilon_deg=0.3,
    smooth_sigma_deg=1.2,
    footprint_radius_deg=6.0,
    show_samples=False,
)

print(f"Saved Cartesian: {Path(vel_out).resolve()}")
print(f"Saved Mollweide: {(Path(vel_out).with_name(Path(vel_out).stem + '_mollweide' + Path(vel_out).suffix)).resolve()}")

In [ ]:
# Velocity map rerun tuned to reduce artificial near-zero regions
from nps_heatmap import run_velocity_map

vel_out = run_velocity_map(
    npz_glob=NPZ_GLOB,
    output_png="nps_velocity_lsr_tuned.png",
    fmin_hz=FMIN_HZ,
    fmax_hz=FMAX_HZ,
    spectrum_mode=MODE,
    grid_step_deg=1.0,
    lmin=LMIN,
    lmax=LMAX,
    bmin=BMIN,
    bmax=BMAX,
    rfi_sigma_clip=RFI_SIGMA,
    rfi_edge_trim=RFI_EDGE_TRIM,
    rfi_patch=RFI_PATCH,
    rfi_ranges_hz=RFI_RANGES_HZ,
    baseline_width=151,
    rest_freq_hz=1420.40575177e6,
    apply_lsr=True,
    obs_lat_deg=37.9183,
    obs_lon_deg=-122.1537,
    obs_alt_m=304.0,
    interpolation_method="linear",
    epsilon_deg=0.3,
    smooth_sigma_deg=0.8,
    footprint_radius_deg=0.0,      # disable hard masking
    velocity_estimator="centroid", # more stable than single-channel peak
    show_samples=False,
)

print(f"Saved Cartesian: {Path(vel_out).resolve()}")
print(f"Saved Mollweide: {(Path(vel_out).with_name(Path(vel_out).stem + '_mollweide' + Path(vel_out).suffix)).resolve()}")

In [ ]:
# Velocity map with less aggressive LSR sign convention
# If map still trends too negative, try lsr_correction_sign=0.0 (diagnostic no-LSR)
# or +1.0 (opposite convention).

vel_out = run_velocity_map(
    npz_glob=NPZ_GLOB,
    output_png="nps_velocity_lsr_signfix.png",
    fmin_hz=FMIN_HZ,
    fmax_hz=FMAX_HZ,
    spectrum_mode=MODE,
    grid_step_deg=1.0,
    lmin=LMIN,
    lmax=LMAX,
    bmin=BMIN,
    bmax=BMAX,
    rfi_sigma_clip=RFI_SIGMA,
    rfi_edge_trim=RFI_EDGE_TRIM,
    rfi_patch=RFI_PATCH,
    rfi_ranges_hz=RFI_RANGES_HZ,
    baseline_width=151,
    rest_freq_hz=1420.40575177e6,
    apply_lsr=True,
    obs_lat_deg=37.9183,
    obs_lon_deg=-122.1537,
    obs_alt_m=304.0,
    interpolation_method="linear",
    epsilon_deg=0.3,
    smooth_sigma_deg=0.8,
    footprint_radius_deg=0.0,
    velocity_estimator="centroid",
    lsr_correction_sign=-1.0,
    show_samples=False,
)

print(f"Saved Cartesian: {Path(vel_out).resolve()}")
print(f"Saved Mollweide: {(Path(vel_out).with_name(Path(vel_out).stem + '_mollweide' + Path(vel_out).suffix)).resolve()}")

In [ ]:
# Velocity map with physically correct LSR correction
# (topocentric -> heliocentric + solar peculiar motion projection)

vel_out = run_velocity_map(
    npz_glob=NPZ_GLOB,
    output_png="nps_velocity_lsr_corrected.png",
    fmin_hz=FMIN_HZ,
    fmax_hz=FMAX_HZ,
    spectrum_mode=MODE,
    grid_step_deg=1.0,
    lmin=LMIN,
    lmax=LMAX,
    bmin=BMIN,
    bmax=BMAX,
    rfi_sigma_clip=RFI_SIGMA,
    rfi_edge_trim=RFI_EDGE_TRIM,
    rfi_patch=RFI_PATCH,
    rfi_ranges_hz=RFI_RANGES_HZ,
    baseline_width=151,
    rest_freq_hz=1420.40575177e6,
    apply_lsr=True,
    obs_lat_deg=37.9183,
    obs_lon_deg=-122.1537,
    obs_alt_m=304.0,
    interpolation_method="linear",
    epsilon_deg=0.3,
    smooth_sigma_deg=0.8,
    footprint_radius_deg=0.0,
    velocity_estimator="centroid",
    show_samples=False,
)

print(f"Saved Cartesian: {Path(vel_out).resolve()}")
print(f"Saved Mollweide: {(Path(vel_out).with_name(Path(vel_out).stem + '_mollweide' + Path(vel_out).suffix)).resolve()}")

In [ ]:
# LSR sanity run: use velocity_source='A' (not fs) for stable Doppler centers
from pathlib import Path
import importlib
import nps_heatmap

importlib.reload(nps_heatmap)
run_velocity_map = nps_heatmap.run_velocity_map

vel_out = run_velocity_map(
    npz_glob=NPZ_GLOB,
    output_png="nps_velocity_lsr_Asource.png",
    fmin_hz=FMIN_HZ,
    fmax_hz=FMAX_HZ,
    spectrum_mode=MODE,
    grid_step_deg=1.0,
    lmin=LMIN,
    lmax=LMAX,
    bmin=BMIN,
    bmax=BMAX,
    rfi_sigma_clip=RFI_SIGMA,
    rfi_edge_trim=RFI_EDGE_TRIM,
    rfi_patch=RFI_PATCH,
    rfi_ranges_hz=RFI_RANGES_HZ,
    baseline_width=151,
    rest_freq_hz=1420.40575177e6,
    apply_lsr=True,
    obs_lat_deg=37.9183,
    obs_lon_deg=-122.1537,
    obs_alt_m=304.0,
    interpolation_method="linear",
    smooth_sigma_deg=0.8,
    footprint_radius_deg=6.0,
    velocity_estimator="centroid",
    velocity_source="fs",
    v_search_kms=80.0,
    show_samples=False,
)

print(f"Saved Cartesian: {Path(vel_out).resolve()}")
print(f"Saved Mollweide: {(Path(vel_out).with_name(Path(vel_out).stem + '_mollweide' + Path(vel_out).suffix)).resolve()}")

In [ ]:
import importlib
import nps_heatmap
importlib.reload(nps_heatmap)
BASELINE_WIDTH = 1024
from nps_heatmap import (load_survey_points, interpolate_map,
                         plot_mollweide_nps_map, _apply_footprint_mask, RfiConfig)
import numpy as np
import matplotlib.pyplot as plt

# ── Config ────────────────────────────────────────────────────────────────────
T_SYS_K   = 43    # confirmed from LAB calibration
# NPZ_GLOB_COMBINED = 'nps_data_4_29_final/*.npz'   # change to combined glob
# If two separate directories:
import glob
all_files = sorted(glob.glob('nps_data_4_29_final/*.npz') + glob.glob('nps_data_5_7/*.npz'))
NPZ_GLOB_COMBINED = all_files  # load_survey_points accepts a list too

rfi_cfg = RfiConfig(
    sigma_clip=RFI_SIGMA,
    edge_trim=RFI_EDGE_TRIM,
    patch_masked=RFI_PATCH,
    manual_ranges_hz=tuple(RFI_RANGES_HZ),
)

# ── Load points ───────────────────────────────────────────────────────────────
points_day1 = load_survey_points(
    npz_glob='nps_data_4_29_final/*.npz',
    fmin_hz=FMIN_HZ,
    fmax_hz=FMAX_HZ,
    spectrum_mode=MODE,
    to_db=False,
    rfi_config=rfi_cfg,
    metric=METRIC,
    baseline_width=BASELINE_WIDTH,
)

points_day2 = load_survey_points(
    npz_glob='nps_data_5_7/*.npz',   # replace with your actual directory name
    fmin_hz=FMIN_HZ,
    fmax_hz=FMAX_HZ,
    spectrum_mode=MODE,
    to_db=False,
    rfi_config=rfi_cfg,
    metric=METRIC,
    baseline_width=BASELINE_WIDTH,
)

points = points_day1 + points_day2
print(f"Day 1: {len(points_day1)}  Day 2: {len(points_day2)}  Total: {len(points)}")

# ── Interpolate ───────────────────────────────────────────────────────────────
ll, bb, zz_raw = interpolate_map(
    points,
    lmin=LMIN, lmax=LMAX,
    bmin=BMIN, bmax=BMAX,
    grid_step_deg=GRID_STEP_DEG,
    epsilon_deg=EPSILON_DEG,
)

# ── Footprint mask ────────────────────────────────────────────────────────────
keep = _apply_footprint_mask(ll, bb, points, radius_deg=6.0)
zz_raw = np.where(keep, zz_raw, np.nan)

# ── Convert to Kelvin ─────────────────────────────────────────────────────────
zz_K = zz_raw * T_SYS_K
zz_K = np.where(zz_K < 0, np.nan, zz_K)

finite = zz_K[np.isfinite(zz_K)]
vmax_K = float(np.nanpercentile(finite, 98)) if finite.size else T_SYS_K
print(f"T_b range: {np.nanmin(zz_K):.1f} – {np.nanmax(zz_K):.1f} K")
print(f"Colorscale: 0 – {vmax_K:.1f} K  (98th percentile)")

# ── Plot ──────────────────────────────────────────────────────────────────────
plt.rcParams.update({
    'font.family': 'serif', 'font.size': 15,
    'axes.labelsize': 18, 'axes.titlesize': 18,
})

fig_m, ax_m = plot_mollweide_nps_map(
    ll, bb, zz_K, points,
    title=(rf'NPS Supershell — Peak HI Brightness Temperature  '
           rf'($\varepsilon$ = {EPSILON_DEG}°)'),
    cmap='magma',
    vmin=0.0,
    vmax=vmax_K,
    center_l_deg=300.0,
    colorbar_label=rf'$T_b$ (K)   [$T_\mathrm{{sys}}$ = {T_SYS_K:.0f} K]',
    show_samples=False,
)

# ── Override axis ticks for finer spacing ────────────────────────────────────
center_l_deg = 300.0

# Longitude: place ticks every 20° instead of 30°
lon_offsets_deg = np.arange(-160, 161, 30)             # change 20 → 10 for finer
ax_m.set_xticks(np.deg2rad(lon_offsets_deg))
lon_labels = [f'{int((off + center_l_deg) % 360)}°' for off in lon_offsets_deg]
ax_m.set_xticklabels(lon_labels, fontsize=15)

# Latitude: place ticks every 15° instead of 30°
lat_ticks_deg = np.arange(-75, 76, 15)                  # change 15 → 10 for finer
ax_m.set_yticks(np.deg2rad(lat_ticks_deg))
ax_m.set_yticklabels([f'{b}°' for b in lat_ticks_deg], fontsize=15)


# Move longitude tick labels below the equator line
ax_m.tick_params(axis='x', pad=15, labeltop=False, labelbottom=True)
ax_m.xaxis.set_label_position('bottom')

for label in ax_m.get_xticklabels():
    label.set_y(-0.8)   # negative pushes labels downward; tweak between -0.05 and -0.15
# Tighter grid to match the new ticks
ax_m.grid(True, alpha=0.4, lw=0.5)

# (Remove these two blocks entirely)

# Tighter grid to match the new ticks
ax_m.grid(True, alpha=0.4, lw=0.5)

# ── Increase colorbar label + tick font size ─────────────────────────────────
cbar_ax = fig_m.axes[-1]
cbar_ax.tick_params(labelsize=13)
cbar_ax.yaxis.label.set_size(15)

fig_m.savefig('nps_brightness_kelvin_combined.png', dpi=300, bbox_inches='tight')
plt.show()
print("Saved: nps_brightness_kelvin_combined.png")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import median_filter
import glob

C_KMS = 299792.458
F_HI  = 1420.40575177e6
T_SYS_K = 50.0

# ── Pick one low-b and one high-b pointing for comparison ────────────────────
def get_spectrum(fn):
    z  = np.load(fn, allow_pickle=True)
    fA = np.asarray(z['freq_hz_A']).ravel(); n = fA.size
    def gs(k):
        a = np.asarray(z[k], dtype=float)
        return (a[:,0][:n] if a.ndim==2 else a[:n])
    sA = 0.5*(gs('spec_A_pol0')+gs('spec_A_pol1'))
    sB = 0.5*(gs('spec_B_pol0')+gs('spec_B_pol1'))
    m  = min(fA.size, sA.size, sB.size)
    with np.errstate(invalid='ignore', divide='ignore'):
        fs = (sA[:m]-sB[:m])/sB[:m]
    v = C_KMS*(F_HI-fA[:m])/F_HI
    l = float(z['l_deg']); b = float(z['b_deg'])
    return v, fs, l, b

# Find lowest-b and highest-b files
best_low = best_high = None
best_low_b = 999; best_high_b = -999
for fn in sorted(glob.glob(NPZ_GLOB)):
    try:
        z = np.load(fn, allow_pickle=True)
        b = float(z['b_deg'])
        if b < best_low_b:  best_low_b,  best_low  = b,  fn
        if b > best_high_b: best_high_b, best_high = b,  fn
    except Exception: continue

fig, axes = plt.subplots(2, 2, figsize=(11, 7))

for row, fn, label in [(0, best_low,  'Low latitude'),
                        (1, best_high, 'High latitude')]:
    v, fs, l, b = get_spectrum(fn)

    win_80  = (np.abs(v) <= 80)  & np.isfinite(fs)
    win_150 = (np.abs(v) <= 150) & np.isfinite(fs)

    # Baseline with current BASELINE_WIDTH=31
    fs_80   = fs[win_80]
    bl_31   = median_filter(fs_80.astype(float),  size=31,  mode='nearest')
    bl_101  = median_filter(fs_80.astype(float),  size=101, mode='nearest')
    line_31  = fs_80 - bl_31
    line_101 = fs_80 - bl_101

    peak_31  = float(np.nanmax(line_31))  * T_SYS_K
    peak_101 = float(np.nanmax(line_101)) * T_SYS_K

    # Left: raw fs in wide window
    axes[row,0].plot(v[win_150], fs[win_150], lw=0.8, color='steelblue')
    axes[row,0].axhline(0, color='k', lw=0.5)
    axes[row,0].axvspan(-80, 80, alpha=0.1, color='orange', label='±80 km/s window')
    axes[row,0].set_title(rf'{label}:  $\ell$={l:.0f}°, $b$={b:.0f}°  —  raw $f_s$')
    axes[row,0].set_xlabel(r'$v_\mathrm{topo}$ (km s$^{-1}$)')
    axes[row,0].set_ylabel(r'$f_s$')
    axes[row,0].legend(fontsize=8)
    axes[row,0].grid(True, alpha=0.3)

    # Right: compare two baseline widths
    axes[row,1].plot(v[win_80], line_31,  lw=0.8, color='steelblue',
                     label=rf'width=31   → $T_b$ peak = {peak_31:.1f} K')
    axes[row,1].plot(v[win_80], line_101, lw=0.8, color='firebrick', alpha=0.8,
                     label=rf'width=101  → $T_b$ peak = {peak_101:.1f} K')
    axes[row,1].axhline(0, color='k', lw=0.5)
    axes[row,1].set_title(rf'{label}: baseline width comparison')
    axes[row,1].set_xlabel(r'$v_\mathrm{topo}$ (km s$^{-1}$)')
    axes[row,1].set_ylabel(r'$f_s - f_{s,\mathrm{base}}$')
    axes[row,1].legend(fontsize=8)
    axes[row,1].grid(True, alpha=0.3)

plt.suptitle('Baseline width effect at low vs high galactic latitude', fontsize=12)
plt.tight_layout()
plt.savefig('baseline_diagnostic.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Low-b  ({best_low_b:.0f}°): width=31 → {float(np.nanmax(median_filter((fs[win_80]).astype(float),31,'nearest').__rsub__(fs[win_80])))*T_SYS_K:.1f} K")
print(f"High-b ({best_high_b:.0f}°): width=31 → check plot")

In [ ]:
import importlib
import nps_heatmap
importlib.reload(nps_heatmap)

from nps_heatmap import (load_velocity_points, interpolate_map,
                         plot_mollweide_nps_map, _apply_footprint_mask,
                         _smooth_nan_grid, RfiConfig)
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
BASELINE_WIDTH = 1000
# ── Config ────────────────────────────────────────────────────────────────────
NPZ_GLOB_DAY1 = 'nps_data_4_29_final/*.npz'
NPZ_GLOB_DAY2 = 'nps_data_5_7/*.npz'   # <-- replace with your actual day-2 path

rfi_cfg = RfiConfig(
    sigma_clip=RFI_SIGMA,
    edge_trim=RFI_EDGE_TRIM,
    patch_masked=RFI_PATCH,
    manual_ranges_hz=tuple(RFI_RANGES_HZ),
)

# ── Load both days separately and combine ─────────────────────────────────────
common = dict(
    fmin_hz=FMIN_HZ,
    fmax_hz=FMAX_HZ,
    spectrum_mode=MODE,
    rfi_config=rfi_cfg,
    baseline_width=101,                 # wider baseline preserves broad low-b lines
    rest_freq_hz=1420.40575177e6,
    apply_lsr=True,
    obs_lat_deg=37.9183,
    obs_lon_deg=-122.1537,
    obs_alt_m=304.0,
    estimator='centroid',
    velocity_source='fs',
    v_search_kms=120.0,                 # widened to capture galactic-plane emission
)

points_day1 = load_velocity_points(npz_glob=NPZ_GLOB_DAY1, **common)
points_day2 = load_velocity_points(npz_glob=NPZ_GLOB_DAY2, **common)
points = points_day1 + points_day2

print(f"Day 1: {len(points_day1)} pointings")
print(f"Day 2: {len(points_day2)} pointings")
print(f"Combined: {len(points)} pointings")

# ── Interpolate ───────────────────────────────────────────────────────────────
ll, bb, zz = interpolate_map(
    points,
    lmin=LMIN, lmax=LMAX,
    bmin=BMIN, bmax=BMAX,
    grid_step_deg=GRID_STEP_DEG,
    epsilon_deg=0.3,
    method='linear',
)

# Smooth
sigma_pix = 0.8 / max(GRID_STEP_DEG, 1e-6)
zz = _smooth_nan_grid(zz, sigma_pix=sigma_pix)

# Mask the unobserved gap
keep = _apply_footprint_mask(ll, bb, points, radius_deg=6.0)
zz   = np.where(keep, zz, np.nan)

# ── Color scaling: symmetric around 0 km/s for diverging map ──────────────────
finite = zz[np.isfinite(zz)]
vmag   = float(np.nanpercentile(np.abs(finite), 95)) if finite.size else 40.0
vmag   = max(vmag, 5.0)   # avoid degenerate range

print(f"v_LSR range: {np.nanmin(zz):+.1f} to {np.nanmax(zz):+.1f} km/s")
print(f"Color scale: ±{vmag:.1f} km/s (95th percentile of |v|)")

# ── Plot Mollweide ────────────────────────────────────────────────────────────
plt.rcParams.update({
    'font.family':    'serif',
    'font.size':      15,
    'axes.labelsize': 15,
    'axes.titlesize': 15,
})

fig_m, ax_m = plot_mollweide_nps_map(
    ll, bb, zz, points,
    title=(rf'NPS Supershell — LSR Doppler Velocity Map  '
           rf'($N$ = {len(points)} pointings, 2 nights combined)'),
    cmap='RdBu_r',
    vmin=-vmag,
    vmax=+vmag,
    center_l_deg=300.0,
    colorbar_label=(rf'$v_\mathrm{{LSR}}$ (km s$^{{-1}}$)   '
                    rf'(red = receding, blue = approaching)'),
    show_samples= False
)

# ── Override axis ticks for finer spacing ────────────────────────────────────
center_l_deg = 300.0

lon_offsets_deg = np.arange(-160, 161, 30)              # change 30 → 20 or 10 for finer
ax_m.set_xticks(np.deg2rad(lon_offsets_deg))
lon_labels = [f'{int((off + center_l_deg) % 360)}°' for off in lon_offsets_deg]
ax_m.set_xticklabels(lon_labels, fontsize=15)

lat_ticks_deg = np.arange(-75, 76, 15)                  # change 15 → 10 for finer
ax_m.set_yticks(np.deg2rad(lat_ticks_deg))
ax_m.set_yticklabels([f'{b}°' for b in lat_ticks_deg], fontsize=15)

ax_m.grid(True, alpha=0.4, lw=0.5)

# ── Increase colorbar label + tick font size ─────────────────────────────────
cbar_ax = fig_m.axes[-1]
cbar_ax.tick_params(labelsize=13)
cbar_ax.yaxis.label.set_size(15)
# If your colorbar happens to be horizontal, swap yaxis → xaxis above

fig_m.savefig('nps_velocity_combined_mollweide.png', dpi=300, bbox_inches='tight')
plt.show()
print(f"\nSaved: nps_velocity_combined_mollweide.png")
print(f"Total integration: {len(points)} pointings across 2 nights")

fig_m.savefig('nps_velocity_combined_mollweide.png',
              dpi=300, bbox_inches='tight')
plt.show()

print(f"\nSaved: nps_velocity_combined_mollweide.png")
print(f"Total integration: {len(points)} pointings across 2 nights")

## Hydrogen Amount Estimation


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import median_filter
import glob

# ── Physical constants ────────────────────────────────────────────────────────
C_KMS    = 299792.458
F_HI     = 1420.40575177e6
T_SYS_K  = 50.0
T_SYS_ERR = 10.0           # ±10 K from the LAB calibration scatter
M_H      = 1.6735e-27
PC_TO_M  = 3.0857e16
M_SUN    = 1.989e30

# ── NPS distance assumption ──────────────────────────────────────────────────
D_PC     = 140.0
D_PC_LO  = 100.0           # ±range for distance uncertainty
D_PC_HI  = 200.0
D_M      = D_PC * PC_TO_M

BEAM_FWHM_DEG = 4.0
BEAM_FWHM_ERR = 0.5

# ── Helper: robust spectrum loader handling both 1D and 2D file formats ──────
def gs(z, key, n):
    a = np.asarray(z[key], dtype=float)
    if a.ndim == 2:
        return a[:, 0][:n] if a.shape[1] > 1 else a.ravel()[:n]
    return a[:n]

def integrated_brightness(fn, baseline_width=101, v_window=120.0):
    """Return (∫T_b dv, sigma, l, b) for one pointing.

    sigma is the 1-sigma uncertainty on the integrated brightness from
    radiometer noise propagated through the trapezoidal integration.
    """
    z  = np.load(fn, allow_pickle=True)
    fA = np.asarray(z['freq_hz_A']).ravel()
    n  = fA.size
    sA = 0.5 * (gs(z, 'spec_A_pol0', n) + gs(z, 'spec_A_pol1', n))
    sB = 0.5 * (gs(z, 'spec_B_pol0', n) + gs(z, 'spec_B_pol1', n))
    m  = min(fA.size, sA.size, sB.size)
    with np.errstate(invalid='ignore', divide='ignore'):
        fs = (sA[:m] - sB[:m]) / sB[:m]
    v  = C_KMS * (F_HI - fA[:m]) / F_HI

    win = (np.abs(v) <= v_window) & np.isfinite(fs)
    if win.sum() < 10:
        return None
    fs_w = fs[win]
    v_w  = v[win]

    # Sort by velocity for clean integration
    order = np.argsort(v_w)
    fs_w  = fs_w[order]
    v_w   = v_w[order]

    bl   = median_filter(fs_w.astype(float), size=baseline_width, mode='nearest')
    line = (fs_w - bl) * T_SYS_K
    line_pos = np.clip(line, 0, None)

    # Integrate T_b over velocity (K·km/s)
    Tb_int = float(np.trapezoid(line_pos, v_w))

    # ── Per-channel noise from off-line region ───────────────────────────────
    # Use ±100 to ±120 km/s as a clean noise window
    off = (np.abs(v_w) > 100) & (np.abs(v_w) < 120)
    if off.sum() < 5:
        sigma_Tb = 0.05 * T_SYS_K
    else:
        sigma_Tb = float(np.nanstd((fs_w - bl)[off]) * T_SYS_K)

    # ── Propagate to integral: sigma(∫T_b dv) ≈ sigma_Tb · sqrt(N) · dv ──────
    dv = float(np.median(np.diff(v_w)))
    N_in_line = int(line_pos.size)
    Tb_int_err = sigma_Tb * np.sqrt(N_in_line) * abs(dv)

    return Tb_int, Tb_int_err, float(z['l_deg']), float(z['b_deg'])

# ── Loop over all pointings ──────────────────────────────────────────────────
all_files = sorted(
    glob.glob('nps_data_4_29_final/*.npz') +
    glob.glob('nps_data_5_7/*.npz')
)

results = []
for fn in all_files:
    try:
        out = integrated_brightness(fn)
        if out is not None:
            results.append(out)
    except Exception as e:
        print(f"Skipped {fn.split('/')[-1]}: {e}")
        continue

print(f"Successfully processed {len(results)} of {len(all_files)} files")
if not results:
    raise RuntimeError("No pointings yielded valid integrated brightness — "
                       "check file format and v_window.")

# Unpack into arrays
Tb_int     = np.array([r[0] for r in results])
Tb_int_err = np.array([r[1] for r in results])
ls         = np.array([r[2] for r in results])
bs         = np.array([r[3] for r in results])

# ── Column density per pointing ──────────────────────────────────────────────
N_HI     = 1.823e18 * Tb_int          # cm⁻²
N_HI_err = 1.823e18 * Tb_int_err

# ── Solid angle per pointing ─────────────────────────────────────────────────
fwhm_rad      = np.deg2rad(BEAM_FWHM_DEG)
omega_beam_sr = (np.pi / (4 * np.log(2))) * fwhm_rad**2
omega_err_rel = 2.0 * BEAM_FWHM_ERR / BEAM_FWHM_DEG   # Ω ∝ FWHM²

# ── Atoms per pointing ───────────────────────────────────────────────────────
area_cm2  = (D_M**2 * omega_beam_sr) * 1e4
atoms     = N_HI * area_cm2

# Per-pointing uncertainty (statistical only; T_sys, D, beam are correlated → handled below)
atoms_err_stat = N_HI_err * area_cm2

N_atoms_total = float(np.sum(atoms))

# ── Total mass and full error propagation ────────────────────────────────────
# M_HI = M_H * Σ atoms_i
# Sources of uncertainty (added in quadrature):
#   1) statistical: combined radiometer noise → σ_stat = sqrt(Σ σ_i²)
#   2) T_sys:       all atoms scale linearly with T_sys
#   3) distance:    M ∝ D²        → σ_D/M  = 2 σ_D/D
#   4) beam:        Ω ∝ FWHM²     → σ_Ω/M = 2 σ_FWHM/FWHM

M_kg     = N_atoms_total * M_H
M_Msun   = M_kg / M_SUN

# Statistical (radiometer)
sigma_stat = float(np.sqrt(np.sum(atoms_err_stat**2))) * M_H / M_SUN

# Systematic: T_sys
sigma_Tsys = M_Msun * (T_SYS_ERR / T_SYS_K)

# Systematic: distance — asymmetric, handle separately
M_lo = M_Msun * (D_PC_LO / D_PC)**2
M_hi = M_Msun * (D_PC_HI / D_PC)**2

# Systematic: beam
sigma_beam = M_Msun * omega_err_rel

sigma_total_sym = float(np.sqrt(sigma_stat**2 + sigma_Tsys**2 + sigma_beam**2))

# ── Report ────────────────────────────────────────────────────────────────────
print("=" * 65)
print("HI Mass Estimate — NPS Region (with error propagation)")
print("=" * 65)
print(f"Pointings:              {len(results)}")
print(f"Beam FWHM:              {BEAM_FWHM_DEG}° ± {BEAM_FWHM_ERR}°")
print(f"Distance:               {D_PC} pc  (range {D_PC_LO}–{D_PC_HI} pc)")
print(f"T_sys:                  {T_SYS_K} ± {T_SYS_ERR} K")
print()
print(f"Median ∫T_b dv:         {np.median(Tb_int):.1f} K·km/s")
print(f"Median N_HI:            {np.median(N_HI):.2e} cm⁻²")
print(f"Total HI atoms:         {N_atoms_total:.2e}")
print(f"Total HI mass:          {M_Msun:.1f} M_sun")
print()
print("Error budget on M_HI:")
print(f"  Statistical (radiometer):  ±{sigma_stat:.2f} M_sun "
      f"({sigma_stat/M_Msun*100:.1f}%)")
print(f"  T_sys (±{T_SYS_ERR:.0f} K):           ±{sigma_Tsys:.2f} M_sun "
      f"({T_SYS_ERR/T_SYS_K*100:.1f}%)")
print(f"  Beam (±{BEAM_FWHM_ERR:.1f}°):           ±{sigma_beam:.2f} M_sun "
      f"({omega_err_rel*100:.1f}%)")
print(f"  Quadrature total:          ±{sigma_total_sym:.2f} M_sun "
      f"({sigma_total_sym/M_Msun*100:.1f}%)")
print()
print(f"Distance bracketing (asymmetric):")
print(f"  D = {D_PC_LO:5.0f} pc → M = {M_lo:5.1f} M_sun")
print(f"  D = {D_PC:5.0f} pc → M = {M_Msun:5.1f} M_sun  (adopted)")
print(f"  D = {D_PC_HI:5.0f} pc → M = {M_hi:5.1f} M_sun")
print()
print("=" * 65)
print(f"FINAL: M_HI = {M_Msun:.1f} ± {sigma_total_sym:.1f} M_sun (stat+sys, fixed D)")
print(f"            ≈ {M_Msun:.0f} M_sun, [{M_lo:.0f} – {M_hi:.0f}] over D range")
print("=" * 65)

# ── Plot: column density map + error bars on integrated profile ──────────────
plt.rcParams.update({'font.family': 'serif', 'font.size': 11})
fig, axes = plt.subplots(1, 2, figsize=(13, 5.2))

# Left: column density scatter
sc = axes[0].scatter(ls, bs, c=N_HI, cmap='magma',
                     s=18, edgecolors='none')
cb = fig.colorbar(sc, ax=axes[0])
cb.set_label(r'$N_\mathrm{HI}$ (cm$^{-2}$)')
axes[0].set_xlabel(r'Galactic longitude $\ell$ (deg)')
axes[0].set_ylabel(r'Galactic latitude $b$ (deg)')
axes[0].set_title(r'HI Column Density per Pointing')
axes[0].grid(True, alpha=0.3)
axes[0].invert_xaxis()

# Right: histogram with error band
order = np.argsort(bs)
axes[1].errorbar(bs[order], N_HI[order], yerr=N_HI_err[order],
                 fmt='o', ms=3, lw=0.5, color='steelblue',
                 ecolor='lightsteelblue', alpha=0.7,
                 label=r'$N_\mathrm{HI}$ ± stat error')
axes[1].set_xlabel(r'Galactic latitude $b$ (deg)')
axes[1].set_ylabel(r'$N_\mathrm{HI}$ (cm$^{-2}$)')
axes[1].set_yscale('log')
axes[1].set_title(rf'$N_\mathrm{{HI}}$ vs $b$ — $M_\mathrm{{HI}}$ = '
                  rf'{M_Msun:.0f} ± {sigma_total_sym:.0f} $M_\odot$')
axes[1].legend()
axes[1].grid(True, alpha=0.3, which='both')

plt.tight_layout()
plt.savefig('hi_mass_estimate.png', dpi=200, bbox_inches='tight')
plt.show()
print("Saved: hi_mass_estimate.png")

In [ ]:
# ── Shell isolation: mass attributable to the bright NPS arc only ─────────────
# Strategy: sum only pointings whose peak T_b exceeds a threshold above the
# diffuse foreground level. This isolates the shell-associated emission from
# the all-sky HI background that fills every line of sight.

import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import median_filter
import glob

# Reuse constants and helpers from previous cell — assumes Tb_int, Tb_int_err,
# ls, bs are already populated. Also need peak T_b per pointing for masking.

trapezoid = getattr(np, 'trapezoid', None) or np.trapz

def peak_Tb(fn, baseline_width=101, v_window=120.0):
    """Return peak T_b in K above baseline."""
    z  = np.load(fn, allow_pickle=True)
    fA = np.asarray(z['freq_hz_A']).ravel(); n = fA.size
    sA = 0.5*(gs(z,'spec_A_pol0',n)+gs(z,'spec_A_pol1',n))
    sB = 0.5*(gs(z,'spec_B_pol0',n)+gs(z,'spec_B_pol1',n))
    m  = min(fA.size, sA.size, sB.size)
    with np.errstate(invalid='ignore', divide='ignore'):
        fs = (sA[:m]-sB[:m])/sB[:m]
    v  = C_KMS*(F_HI-fA[:m])/F_HI
    win = (np.abs(v) <= v_window) & np.isfinite(fs)
    if win.sum() < 10: return None, None, None
    fs_w = fs[win]; v_w = v[win]
    bl   = median_filter(fs_w.astype(float), size=baseline_width, mode='nearest')
    line = (fs_w - bl) * T_SYS_K
    return float(np.nanmax(line)), float(z['l_deg']), float(z['b_deg'])

# Recompute peak per file aligned to (ls, bs) order
all_files = sorted(
    glob.glob('nps_data_4_29_final/*.npz') +
    glob.glob('nps_data_5_7/*.npz')
)
pk_lookup = {}
for fn in all_files:
    try:
        pk, l, b = peak_Tb(fn)
        if pk is not None:
            pk_lookup[(round(l,2), round(b,2))] = pk
    except Exception:
        continue

Tb_peak = np.array([pk_lookup.get((round(l,2), round(b,2)), np.nan)
                    for l, b in zip(ls, bs)])

# ── Determine background level and shell threshold ───────────────────────────
# Use the median peak T_b at high latitudes (b > 70°) as the diffuse foreground
foreground_mask = bs > 70
T_bg     = float(np.nanmedian(Tb_peak[foreground_mask]))
T_bg_std = float(np.nanstd(Tb_peak[foreground_mask]))

# Shell threshold: 3σ above the foreground (tune this to your structure)
T_THRESH = T_bg + 3 * T_bg_std

print(f"Foreground (b > 70°):  T_bg  = {T_bg:.2f} K  (σ = {T_bg_std:.2f} K)")
print(f"Shell threshold:        T > {T_THRESH:.2f} K  (3σ above foreground)")

# ── Create the shell mask and subtract foreground from each pointing ─────────
shell_mask = Tb_peak > T_THRESH
print(f"\nShell pointings:       {shell_mask.sum()} / {len(Tb_peak)}")

# Estimate foreground integrated brightness (∫T_bg dv across line width)
# Use median ∫T_b dv at high-b pointings as the foreground level
Tb_int_bg = float(np.nanmedian(Tb_int[foreground_mask]))
print(f"Foreground ∫T_b dv:    {Tb_int_bg:.1f} K·km/s")

# Excess (shell-only) integrated brightness — clipped to zero
Tb_int_excess = np.clip(Tb_int - Tb_int_bg, 0, None)
Tb_int_excess[~shell_mask] = 0    # zero out non-shell pointings

# ── Recompute mass for shell-only emission ───────────────────────────────────
N_HI_shell = 1.823e18 * Tb_int_excess

fwhm_rad      = np.deg2rad(BEAM_FWHM_DEG)
omega_beam_sr = (np.pi / (4 * np.log(2))) * fwhm_rad**2
area_cm2      = (D_M**2 * omega_beam_sr) * 1e4

atoms_shell    = N_HI_shell * area_cm2
N_atoms_shell  = float(np.sum(atoms_shell))
M_shell_kg     = N_atoms_shell * M_H
M_shell_Msun   = M_shell_kg / M_SUN

# Error propagation (same scheme as before, applied to the shell mass)
sigma_stat_shell = float(np.sqrt(np.sum((1.823e18 * Tb_int_err[shell_mask] *
                                          area_cm2)**2))) * M_H / M_SUN
sigma_Tsys_shell = M_shell_Msun * (T_SYS_ERR / T_SYS_K)
sigma_beam_shell = M_shell_Msun * (2 * BEAM_FWHM_ERR / BEAM_FWHM_DEG)
sigma_shell_total = float(np.sqrt(sigma_stat_shell**2 +
                                   sigma_Tsys_shell**2 +
                                   sigma_beam_shell**2))

# Distance bracketing
M_shell_lo = M_shell_Msun * (D_PC_LO / D_PC)**2
M_shell_hi = M_shell_Msun * (D_PC_HI / D_PC)**2

# ── Report ────────────────────────────────────────────────────────────────────
print()
print("=" * 65)
print("Shell-Isolated HI Mass (NPS arc only)")
print("=" * 65)
print(f"Foreground subtracted:  {Tb_int_bg:.1f} K·km/s per pointing")
print(f"Shell pointings:        {shell_mask.sum()}")
print(f"Shell HI atoms:         {N_atoms_shell:.2e}")
print(f"Shell HI mass:          {M_shell_Msun:.1f} M_sun  (D = {D_PC:.0f} pc)")
print()
print("Shell mass error budget:")
print(f"  Statistical:           ±{sigma_stat_shell:.2f} M_sun")
print(f"  T_sys:                 ±{sigma_Tsys_shell:.2f} M_sun")
print(f"  Beam:                  ±{sigma_beam_shell:.2f} M_sun")
print(f"  Quadrature total:      ±{sigma_shell_total:.2f} M_sun "
      f"({sigma_shell_total/M_shell_Msun*100:.1f}%)")
print()
print(f"Distance bracketing:")
print(f"  D = {D_PC_LO:5.0f} pc → M_shell = {M_shell_lo:6.1f} M_sun")
print(f"  D = {D_PC:5.0f} pc → M_shell = {M_shell_Msun:6.1f} M_sun  (adopted)")
print(f"  D = {D_PC_HI:5.0f} pc → M_shell = {M_shell_hi:6.1f} M_sun")
print()
print("=" * 65)
print(f"Total HI in survey footprint: {M_Msun:.0f} ± {sigma_total_sym:.0f} M_sun")
print(f"NPS shell-only HI mass:       {M_shell_Msun:.0f} ± "
      f"{sigma_shell_total:.0f} M_sun")
print(f"Shell fraction of total:      {M_shell_Msun/M_Msun*100:.1f}%")
print("=" * 65)

# ── Visualisation: peak T_b map with shell mask overlaid ─────────────────────
plt.rcParams.update({'font.family': 'serif', 'font.size': 11})
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

# Left: peak T_b with threshold contour
sc1 = axes[0].scatter(ls, bs, c=Tb_peak, cmap='magma',
                       s=22, edgecolors='none', vmin=0, vmax=np.nanpercentile(Tb_peak, 98))
cb1 = fig.colorbar(sc1, ax=axes[0])
cb1.set_label(r'Peak $T_b$ (K)')
axes[0].scatter(ls[shell_mask], bs[shell_mask],
                facecolors='none', edgecolors='cyan', s=45, lw=0.8,
                label=rf'Shell mask ($T_b$ > {T_THRESH:.1f} K)')
axes[0].set_xlabel(r'Galactic longitude $\ell$ (deg)')
axes[0].set_ylabel(r'Galactic latitude $b$ (deg)')
axes[0].set_title('Peak HI brightness with NPS shell mask')
axes[0].invert_xaxis()
axes[0].legend(loc='lower left', framealpha=0.9, fontsize=9)
axes[0].grid(True, alpha=0.3)

# Right: foreground-subtracted column density (shell only)
sc2 = axes[1].scatter(ls[shell_mask], bs[shell_mask],
                       c=N_HI_shell[shell_mask], cmap='viridis',
                       s=22, edgecolors='none')
cb2 = fig.colorbar(sc2, ax=axes[1])
cb2.set_label(r'Shell $N_\mathrm{HI}$ (cm$^{-2}$)')
axes[1].set_xlabel(r'Galactic longitude $\ell$ (deg)')
axes[1].set_ylabel(r'Galactic latitude $b$ (deg)')
axes[1].set_title(rf'Shell-only column density'
                  rf' — $M_\mathrm{{shell}}$ = {M_shell_Msun:.0f} ± '
                  rf'{sigma_shell_total:.0f} $M_\odot$')
axes[1].set_xlim(axes[0].get_xlim())
axes[1].set_ylim(axes[0].get_ylim())
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('hi_mass_shell.png', dpi=200, bbox_inches='tight')
plt.show()
print("\nSaved: hi_mass_shell.png")

In [ ]:
import numpy as np

# ── 1. Threshold sensitivity ─────────────────────────────────────────────────
print("=== Shell mass vs threshold ===")
print(f"{'Threshold (σ)':>15}  {'T_thresh (K)':>13}  {'N_pointings':>12}"
      f"  {'M_shell (M_sun)':>16}")
print("-" * 60)

masses_vs_thresh = []
for n_sigma in [1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 5.0]:
    thr = T_bg + n_sigma * T_bg_std
    mask = Tb_peak > thr
    if not mask.any(): continue
    excess = np.clip(Tb_int - Tb_int_bg, 0, None)
    excess[~mask] = 0
    N = 1.823e18 * excess
    atoms = N * area_cm2
    M = float(np.sum(atoms)) * M_H / M_SUN
    masses_vs_thresh.append((n_sigma, thr, mask.sum(), M))
    print(f"{n_sigma:>15.1f}  {thr:>13.2f}  {mask.sum():>12}  {M:>16.0f}")

# ── 2. Foreground robustness ─────────────────────────────────────────────────
print("\n=== Foreground sensitivity (test different b cuts) ===")
print(f"{'b > X°':>10}  {'N pointings':>12}  {'T_bg (K)':>10}"
      f"  {'∫T_bg dv':>12}  {'M_shell':>10}")
print("-" * 60)

for b_cut in [60, 65, 70, 75, 80, 85]:
    fg = bs > b_cut
    if fg.sum() < 5: continue
    Tbg_alt   = float(np.nanmedian(Tb_peak[fg]))
    TbgInt_alt = float(np.nanmedian(Tb_int[fg]))
    thr_alt    = Tbg_alt + 3 * float(np.nanstd(Tb_peak[fg]))
    mask_alt   = Tb_peak > thr_alt
    excess_alt = np.clip(Tb_int - TbgInt_alt, 0, None)
    excess_alt[~mask_alt] = 0
    M_alt = float(np.sum(1.823e18 * excess_alt * area_cm2)) * M_H / M_SUN
    print(f"  b > {b_cut:>3d}°  {fg.sum():>12}  {Tbg_alt:>10.2f}"
          f"  {TbgInt_alt:>12.1f}  {M_alt:>10.0f}")

# ── 3. Spatial completeness check ────────────────────────────────────────────
# What fraction of the *shell area* did we actually observe?
# Shell defined as: bright pixels (Tb_peak > T_THRESH) form a region;
# we need to know if our footprint covers that region or cuts it off.

print("\n=== Spatial coverage of NPS shell ===")
print(f"Shell extent in your data:")
print(f"  ℓ range: {ls[shell_mask].min():.1f}° – {ls[shell_mask].max():.1f}°")
print(f"  b range: {bs[shell_mask].min():.1f}° – {bs[shell_mask].max():.1f}°")
print(f"\nClassical NPS extent (Heiles 1998):")
print(f"  ℓ range: ~280° – 30°  (across galactic centre)")
print(f"  b range: ~0° – 80°")

# Check if shell pointings touch the survey edge — indicates truncation
on_edge = (
    (np.abs(ls[shell_mask] - LMIN) < 5) |
    (np.abs(ls[shell_mask] - LMAX) < 5) |
    (np.abs(bs[shell_mask] - BMIN) < 5) |
    (np.abs(bs[shell_mask] - BMAX) < 5)
)
print(f"\nShell pointings on survey edge: {on_edge.sum()} / {shell_mask.sum()}"
      f"  ({on_edge.sum()/shell_mask.sum()*100:.0f}%)")
if on_edge.sum() > 5:
    print("  → Shell is truncated by survey footprint; M_shell is a lower limit.")
else:
    print("  → Shell fits within survey footprint.")

In [ ]:
# ── Shell isolation — physically motivated threshold ──────────────────────────
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import median_filter
import glob

trapezoid = getattr(np, 'trapezoid', None) or np.trapz

def peak_Tb(fn, baseline_width=101, v_window=120.0):
    z  = np.load(fn, allow_pickle=True)
    try:
        l = float(z['l_deg']); b = float(z['b_deg'])
    except (TypeError, ValueError):
        return None, None, None
    fA = np.asarray(z['freq_hz_A']).ravel(); n = fA.size
    sA = 0.5*(gs(z,'spec_A_pol0',n)+gs(z,'spec_A_pol1',n))
    sB = 0.5*(gs(z,'spec_B_pol0',n)+gs(z,'spec_B_pol1',n))
    m  = min(fA.size, sA.size, sB.size)
    with np.errstate(invalid='ignore', divide='ignore'):
        fs = (sA[:m]-sB[:m])/sB[:m]
    v  = C_KMS*(F_HI-fA[:m])/F_HI
    win = (np.abs(v) <= v_window) & np.isfinite(fs)
    if win.sum() < 10: return None, None, None
    fs_w = fs[win]
    bl   = median_filter(fs_w.astype(float), size=baseline_width, mode='nearest')
    line = (fs_w - bl) * T_SYS_K
    return float(np.nanmax(line)), l, b

# ── Recompute peak T_b per pointing ──────────────────────────────────────────
all_files = sorted(
    glob.glob('nps_data_4_29_final/*.npz') +
    glob.glob('nps_data_5_7/*.npz')
)
pk_lookup = {}
for fn in all_files:
    try:
        pk, l, b = peak_Tb(fn)
        if pk is not None:
            pk_lookup[(round(l,2), round(b,2))] = pk
    except Exception:
        continue

Tb_peak = np.array([pk_lookup.get((round(l,2), round(b,2)), np.nan)
                    for l, b in zip(ls, bs)])

# ── Foreground: use only b > 80° AND outside the known NPS longitude range ───
# NPS arc sits roughly l = 320–30° (wrapping) and b = 10–80°
# Use truly blank-sky pointings for the foreground estimate
nps_lon_mask = ((ls > 30) & (ls < 320))   # outside NPS longitude range
foreground_mask = (bs > 75) & nps_lon_mask
print(f"Foreground pointings (b>75°, outside NPS lon): {foreground_mask.sum()}")

if foreground_mask.sum() < 5:
    # Fallback: just use b > 75°
    foreground_mask = bs > 75
    print(f"  Fallback to b>75°: {foreground_mask.sum()} pointings")

T_bg      = float(np.nanmedian(Tb_peak[foreground_mask]))
T_bg_std  = float(np.nanstd(Tb_peak[foreground_mask]))
Tb_int_bg = float(np.nanmedian(Tb_int[foreground_mask]))

print(f"Foreground T_bg       = {T_bg:.2f} K  (σ = {T_bg_std:.2f} K)")
print(f"Foreground ∫T_b dv    = {Tb_int_bg:.1f} K·km/s")

# ── Threshold sweep to find stable region ────────────────────────────────────
print(f"\n{'σ':>5}  {'T_thresh':>10}  {'N_shell':>8}  {'M_shell':>12}")
print("-" * 42)
sweep_results = []
for n_sigma in [1.0, 1.5, 2.0, 2.5, 3.0]:
    thr = T_bg + n_sigma * T_bg_std
    mask = Tb_peak > thr
    excess = np.clip(Tb_int - Tb_int_bg, 0, None)
    excess[~mask] = 0
    M = float(np.sum(1.823e18 * excess * area_cm2)) * M_H / M_SUN
    sweep_results.append((n_sigma, thr, mask.sum(), M))
    print(f"{n_sigma:>5.1f}  {thr:>10.1f}  {mask.sum():>8}  {M:>12.0f}")

# ── Adopt 2σ threshold as the report value ───────────────────────────────────
# 2σ captures the NPS arc more completely while remaining statistically
# meaningful; 3σ is too restrictive given our noise floor.
N_SIGMA_ADOPT = 2.0
T_THRESH = T_bg + N_SIGMA_ADOPT * T_bg_std
shell_mask = Tb_peak > T_THRESH

print(f"\nAdopted threshold: {N_SIGMA_ADOPT}σ → T > {T_THRESH:.1f} K")
print(f"Shell pointings:   {shell_mask.sum()} / {len(Tb_peak)}")

# ── Shell mass ────────────────────────────────────────────────────────────────
Tb_int_excess = np.clip(Tb_int - Tb_int_bg, 0, None)
Tb_int_excess[~shell_mask] = 0

N_HI_shell   = 1.823e18 * Tb_int_excess
atoms_shell   = N_HI_shell * area_cm2
M_shell_Msun  = float(np.sum(atoms_shell)) * M_H / M_SUN

sigma_stat_shell = float(np.sqrt(np.sum(
    (1.823e18 * Tb_int_err[shell_mask] * area_cm2)**2
))) * M_H / M_SUN
sigma_Tsys_shell = M_shell_Msun * (T_SYS_ERR / T_SYS_K)
sigma_beam_shell = M_shell_Msun * (2 * BEAM_FWHM_ERR / BEAM_FWHM_DEG)
sigma_shell_total = float(np.sqrt(
    sigma_stat_shell**2 + sigma_Tsys_shell**2 + sigma_beam_shell**2
))

M_shell_lo = M_shell_Msun * (D_PC_LO / D_PC)**2
M_shell_hi = M_shell_Msun * (D_PC_HI / D_PC)**2

print(f"\nShell mass:  {M_shell_Msun:.0f} ± {sigma_shell_total:.0f} M_sun")
print(f"  D=100 pc:  {M_shell_lo:.0f} M_sun")
print(f"  D=140 pc:  {M_shell_Msun:.0f} M_sun")
print(f"  D=200 pc:  {M_shell_hi:.0f} M_sun")

# ── Plots ─────────────────────────────────────────────────────────────────────
plt.rcParams.update({'font.family': 'serif', 'font.size': 11})
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

# Left: peak T_b map with shell mask
# Left: peak T_b map — grey for non-shell, colored for shell
axes[0].scatter(ls[~shell_mask], bs[~shell_mask],
                c='0.75', s=18, edgecolors='none', alpha=0.5,
                label='Non-shell')
sc1s = axes[0].scatter(ls[shell_mask], bs[shell_mask],
                        c=Tb_peak[shell_mask], cmap='magma',
                        s=40, edgecolors='none',
                        vmin=0, vmax=vmax_pk,
                        label=rf'Shell ($T_b$ > {T_THRESH:.1f} K)')
cb1 = fig.colorbar(sc1s, ax=axes[0])
cb1.set_label(r'Peak $T_b$ (K)')

# Right: shell column density
sc2 = axes[1].scatter(ls[shell_mask], bs[shell_mask],
                       c=N_HI_shell[shell_mask], cmap='viridis',
                       s=22, edgecolors='none')
cb2 = fig.colorbar(sc2, ax=axes[1])
cb2.set_label(r'Shell $N_\mathrm{HI}$ (cm$^{-2}$)')
axes[1].set_xlabel(r'Galactic longitude $\ell$ (deg)')
axes[1].set_ylabel(r'Galactic latitude $b$ (deg)')
axes[1].set_title(
    rf'Shell-only $N_\mathrm{{HI}}$ — '
    rf'$M_\mathrm{{shell}}$ = {M_shell_Msun:.0f} $\pm$ {sigma_shell_total:.0f} $M_\odot$'
)
axes[1].set_xlim(axes[0].get_xlim())
axes[1].set_ylim(axes[0].get_ylim())
axes[1].invert_xaxis()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('hi_mass_shell.png', dpi=200, bbox_inches='tight')
plt.show()